# 03. Анализ blacklist-событий и заказов

**Тип проекта:** учебный кейс по SQL и обработке данных  
**Инструменты:** SQLite, SQL, Python, pandas, Jupyter Notebook  
**Статус:** проверка и оформление результатов

## 1. Постановка задачи

### Контекст

В проекте интернет-магазина есть логи обращений к бонусам и промокодам, данные о заказах и список клиентов. SQL-процесс, рассчитывающий пользовательские признаки, содержит ошибки и незавершённые формулы.

### Цель

Исправить SQL-запросы и получить восемь признаков для каждого пользователя из списка `customer_ids`.

### План

1. Посмотреть структуру исходных данных.
2. Выделить настоящие blacklist-срабатывания.
3. Исправить расчёты по логам и заказам.
4. Объединить показатели на уровне пользователя.
5. Проверить итоговую таблицу.


## 2. Знакомство с данными

В работе используются три CSV-файла:

| Таблица | Содержание |
|---|---|
| `logs` | Обращения к бонусам и промокодам |
| `orders` | Заказы, суммы и статусы |
| `customer_ids` | Клиенты, для которых нужны признаки |

Сначала загружу файлы и проверю их размеры, затем посмотрю первые строки.


In [1]:

import pandas as pd
from pathlib import Path

# Папка нашего проекта
project = Path.cwd()

if project.name != "03-blacklist-sql":
    project = project / "03-blacklist-sql"

# Папка с исходными данными
data_dir = project / "originals" / "37890-solution-main" / "data"

# Загружаем таблицы
logs = pd.read_csv(data_dir / "logs.csv")
orders = pd.read_csv(data_dir / "orders.csv")
customers = pd.read_csv(data_dir / "customer_ids.csv")

# Проверяем размеры
print("Логи:", logs.shape)
print("Заказы:", orders.shape)
print("Пользователи:", customers.shape)


Логи: (40000, 9)
Заказы: (8000, 7)
Пользователи: (5000, 1)


### Первые записи

Загрузились **40 000 событий**, **8 000 заказов** и **5 000 пользователей**. Посмотрю названия полей и примеры записей, прежде чем переходить к SQL.


In [2]:

from IPython.display import display

# Первые пять записей логов
print("Логи:")
display(logs.head())

# Первые пять заказов
print("Заказы:")
display(orders.head())

# Первые пять пользователей
print("Пользователи:")
display(customers.head())


Логи:


,log_id,event_dttm,customer_id,session_id,endpoint,promo_id,triggered_rules,triggered_rule_codes,request_status
0,1,2026-07-17 04:10:00,100001,target-100001-1,bonuses,NaN,1.0,"[""blacklist""]",rejected
1,2,2026-07-17 04:12:00,100001,target-100001-1,bonuses,NaN,2.0,"[""blacklist"",""bonus_daily_limit""]",rejected
2,3,2026-07-17 04:14:00,100001,target-100001-1,bonuses,NaN,1.0,"[""blacklist""]",rejected
3,4,2026-07-17 04:28:00,100001,target-100001-1,promo,PROMO104,1.0,"[""blacklist""]",rejected
4,5,2026-07-02 00:36:00,100001,target-100001-2,bonuses,NaN,2.0,"[""blacklist"",""bonus_account_mismatch""]",rejected


Заказы:


,order_id,created_dttm,customer_id,session_id,promo_id,order_amount,order_status
0,1,2026-07-17 06:04:00,100001,target-100001-1,NaN,15373.50,completed
1,2,2026-07-02 00:54:59,100001,target-100001-2,NaN,23410.46,completed
2,3,2026-07-19 05:24:00,100002,target-100002-1,NaN,22334.94,completed
3,4,2026-07-01 18:16:59,100002,target-100002-2,NaN,6161.33,paid
4,5,2026-07-17 19:16:00,100003,target-100003-1,NaN,14426.88,completed


Пользователи:


,customer_id
0,100001
1,100002
2,100003
3,100004
4,100005


**Наблюдение.** В первых записях один клиент обращается и к `bonuses`, и к `promo` в рамках одной сессии. В заказах встречаются разные статусы, а `promo_id` может быть пустым. Эти поля понадобятся для расчёта признаков.

### Подготовка SQLite

В логах есть `customer_id`, `session_id`, тип обращения и список сработавших правил. В заказах — данные об оплате и промокодах. Эти таблицы можно связать по пользователю и сессии.

Для SQL-запросов загружу CSV во **временную SQLite-базу в памяти**. Исходные файлы при этом не меняются. Первый запрос просто покажет пять событий.


In [3]:

import sqlite3

# Создаём временную базу SQLite в памяти
sql_conn = sqlite3.connect(":memory:")

# Загружаем наши три таблицы
logs.to_sql("logs", sql_conn, index=False, if_exists="replace")
orders.to_sql("orders", sql_conn, index=False, if_exists="replace")
customers.to_sql("customer_ids", sql_conn, index=False, if_exists="replace")

# Выполняем первый SQL-запрос
query = """
SELECT
    log_id,
    customer_id,
    session_id,
    endpoint,
    triggered_rule_codes,
    request_status
FROM logs
ORDER BY log_id
LIMIT 5;
"""

result = pd.read_sql_query(query, sql_conn)

result


,log_id,customer_id,session_id,endpoint,triggered_rule_codes,request_status
0,1,100001,target-100001-1,bonuses,"[""blacklist""]",rejected
1,2,100001,target-100001-1,bonuses,"[""blacklist"",""bonus_daily_limit""]",rejected
2,3,100001,target-100001-1,bonuses,"[""blacklist""]",rejected
3,4,100001,target-100001-1,promo,"[""blacklist""]",rejected
4,5,100001,target-100001-2,bonuses,"[""blacklist"",""bonus_account_mismatch""]",rejected


**Результат.** Первый `SELECT` вернул пять записей из `logs`. Теперь можно переходить от просмотра данных к фильтрации событий.

## 3. Поиск blacklist-событий

### Предварительная фильтрация

Сначала выберу обращения к `bonuses`, которые были отклонены и где сработало хотя бы одно правило. Но это ещё не означает, что сработал именно `blacklist`.


In [4]:

# Выбираем отклонённые обращения к бонусам
query = """
SELECT
    log_id,
    customer_id,
    session_id,
    endpoint,
    triggered_rules,
    triggered_rule_codes,
    request_status
FROM logs
WHERE endpoint = 'bonuses'
  AND request_status = 'rejected'
  AND triggered_rules > 0
ORDER BY log_id
LIMIT 10;
"""

filtered_logs = pd.read_sql_query(query, sql_conn)

filtered_logs


,log_id,customer_id,session_id,endpoint,triggered_rules,triggered_rule_codes,request_status
0,1,100001,target-100001-1,bonuses,1.0,"[""blacklist""]",rejected
1,2,100001,target-100001-1,bonuses,2.0,"[""blacklist"",""bonus_daily_limit""]",rejected
2,3,100001,target-100001-1,bonuses,1.0,"[""blacklist""]",rejected
3,5,100001,target-100001-2,bonuses,2.0,"[""blacklist"",""bonus_account_mismatch""]",rejected
4,6,100001,target-100001-2,bonuses,2.0,"[""blacklist"",""bonus_account_mismatch""]",rejected
5,7,100001,target-100001-2,bonuses,2.0,"[""blacklist"",""bonus_daily_limit""]",rejected
6,9,100002,target-100002-1,bonuses,1.0,"[""blacklist""]",rejected
7,11,100002,target-100002-2,bonuses,1.0,"[""blacklist""]",rejected
8,13,100003,target-100003-1,bonuses,2.0,"[""blacklist"",""suspicious_bonus_activity""]",rejected
9,15,100003,target-100003-2,bonuses,2.0,"[""blacklist"",""bonus_daily_limit""]",rejected


### Сколько таких обращений всего?

Первые строки позволяют увидеть примеры. Теперь посчитаю **все** отклонённые обращения к `bonuses` с хотя бы одним сработавшим правилом через `COUNT(*)`.

In [5]:

query = """
SELECT COUNT(*) AS rejected_bonus_requests
FROM logs
WHERE endpoint = 'bonuses'
  AND request_status = 'rejected'
  AND triggered_rules > 0;
"""

pd.read_sql_query(query, sql_conn)


,rejected_bonus_requests
0,9800


**Результат.** Получилось **9 800** отклонённых обращений к `bonuses` с хотя бы одним сработавшим правилом. Теперь уточню, в каких из них присутствует конкретно `blacklist`.


### Проверка конкретного правила

В поле `triggered_rule_codes` хранится JSON-массив. Через `json_each()` проверю три условия: число сработавших правил больше нуля, статус `rejected` и наличие `blacklist` в списке правил. Подсчёт сделаю отдельно по типам обращения.


In [6]:

query = """
SELECT
    endpoint,
    COUNT(*) AS blacklist_requests
FROM logs AS l
WHERE l.triggered_rules > 0
  AND l.request_status = 'rejected'
  AND EXISTS (
      SELECT 1
      FROM json_each(l.triggered_rule_codes) AS rule
      WHERE rule.value = 'blacklist'
  )
GROUP BY endpoint
ORDER BY endpoint;
"""

blacklist_stats = pd.read_sql_query(query, sql_conn)

blacklist_stats


,endpoint,blacklist_requests
0,bonuses,3065
1,promo,2313


**Результат.** Настоящих blacklist-срабатываний оказалось **3 065** для `bonuses` и **2 313** для `promo`.

Это количество **событий**, а не сессий. Дальше сгруппирую записи по пользователю и сессии.


## 4. Признаки по логам: `01_logs.sql`

### Группировка по сессиям

Один пользователь может сделать несколько запросов в одной сессии. Поэтому сначала посчитаю blacklist-срабатывания отдельно для каждой пары `customer_id` и `session_id`.

Для такого подсчёта в SQL используются `GROUP BY` и `SUM(CASE WHEN ...)`.


In [7]:

query = """
WITH blacklist_events AS (
    SELECT
        customer_id,
        session_id,
        endpoint
    FROM logs AS l
    WHERE triggered_rules > 0
      AND request_status = 'rejected'
      AND EXISTS (
          SELECT 1
          FROM json_each(l.triggered_rule_codes) AS rule
          WHERE rule.value = 'blacklist'
      )
)

SELECT
    customer_id,
    session_id,

    SUM(CASE
        WHEN endpoint = 'bonuses' THEN 1
        ELSE 0
    END) AS bonus_blacklist_requests_cnt,

    SUM(CASE
        WHEN endpoint = 'promo' THEN 1
        ELSE 0
    END) AS promo_blacklist_requests_cnt

FROM blacklist_events
GROUP BY customer_id, session_id
ORDER BY customer_id, session_id
LIMIT 10;
"""

session_stats = pd.read_sql_query(query, sql_conn)

session_stats


,customer_id,session_id,bonus_blacklist_requests_cnt,promo_blacklist_requests_cnt
0,100001,target-100001-1,3,1
1,100001,target-100001-2,3,1
2,100002,log-only-004201,1,0
3,100002,target-100002-1,1,1
4,100002,target-100002-2,1,1
5,100003,target-100003-1,1,1
6,100003,target-100003-2,3,1
7,100004,log-only-002575,0,1
8,100004,target-100004-1,1,1
9,100004,target-100004-2,3,1


**Результат группировки.** Например, у пользователя `100001` в сессии `target-100001-1` найдено **3** blacklist-срабатывания в `bonuses` и **1** в `promo`. При группировке другие сессии того же клиента считаются отдельно.

### Время событий

Помимо количества, важен **порядок**: сначала blacklist в `bonuses`, потом blacklist в `promo` в той же сессии. Для начала посмотрю самые ранние события каждого типа через `MIN()`.


In [8]:

query = """
SELECT
    l.customer_id,
    l.session_id,

    MIN(CASE
        WHEN l.endpoint = 'bonuses'
        THEN l.event_dttm
    END) AS first_bonus,

    MIN(CASE
        WHEN l.endpoint = 'promo'
        THEN l.event_dttm
    END) AS first_promo

FROM logs AS l

WHERE l.triggered_rules > 0
  AND l.request_status = 'rejected'
  AND EXISTS (
      SELECT 1
      FROM json_each(l.triggered_rule_codes) AS rule
      WHERE rule.value = 'blacklist'
  )

GROUP BY l.customer_id, l.session_id
ORDER BY l.customer_id, l.session_id
LIMIT 10;
"""

session_times = pd.read_sql_query(query, sql_conn)

session_times


,customer_id,session_id,first_bonus,first_promo
0,100001,target-100001-1,2026-07-17 04:10:00,2026-07-17 04:28:00
1,100001,target-100001-2,2026-07-02 00:36:00,2026-07-02 00:55:00
2,100002,log-only-004201,2026-07-08 16:45:55,None
3,100002,target-100002-1,2026-07-19 01:24:00,2026-07-19 01:36:00
4,100002,target-100002-2,2026-07-01 18:11:00,2026-07-01 18:17:00
5,100003,target-100003-1,2026-07-17 19:02:00,2026-07-17 19:14:00
6,100003,target-100003-2,2026-07-01 00:11:00,2026-07-01 00:25:00
7,100004,log-only-002575,None,2026-07-04 01:18:15
8,100004,target-100004-1,2026-07-13 11:03:00,2026-07-13 11:15:00
9,100004,target-100004-2,2026-07-27 07:16:00,2026-07-27 07:29:00


**Наблюдение.** Например, в сессии `target-100001-1` первое срабатывание в `bonuses` было в 04:10, а в `promo` — в 04:28. Есть и сессии, где один из типов событий отсутствует.

Для полной проверки порядка недостаточно просто сравнивать два первых события: `promo` мог встретиться до `bonuses`, а затем повториться после него.


### Исправление и запуск `01_logs.sql`

В рабочей копии файла исправила синтаксис и заполнила `blacklist_funnel_flg`. Этот признак равен 1 только тогда, когда **после** blacklist-срабатывания в `bonuses` найдено blacklist-срабатывание в `promo` в той же сессии.

Запущу SQL-файл в SQLite и посмотрю несколько строк результата.


In [9]:

# Запускаем исправленный SQL-файл
sql_file = project / "sql" / "01_logs.sql"

sql_conn.executescript(
    sql_file.read_text(encoding="utf-8")
)

# Проверяем результат
query = """
SELECT
    customer_id,
    session_id,
    bonus_blacklist_requests_cnt,
    promo_blacklist_requests_cnt,
    blacklist_funnel_flg
FROM logs_features
ORDER BY customer_id, session_id
LIMIT 10;
"""

pd.read_sql_query(query, sql_conn)


,customer_id,session_id,bonus_blacklist_requests_cnt,promo_blacklist_requests_cnt,blacklist_funnel_flg
0,100001,log-only-001278,0,0,0
1,100001,log-only-002313,0,0,0
2,100001,log-only-003719,0,0,0
3,100001,log-only-004156,0,0,0
4,100001,log-only-005736,0,0,0
5,100001,log-only-008241,0,0,0
6,100001,log-only-011242,0,0,0
7,100001,log-only-011679,0,0,0
8,100001,target-100001-1,3,1,1
9,100001,target-100001-2,3,1,1


**Наблюдение.** В `logs_features` есть и сессии с нулевыми значениями — это нормально: исходный SQL работает со всеми сессиями логов, не только с blacklist-событиями.

### Проверка флага воронки

После создания `logs_features` проверю, сколько сессий получили флаг 1 и нет ли среди них случаев с пустыми датами или неверным порядком событий.

In [10]:

query = """
SELECT
    COUNT(*) AS total_sessions,
    SUM(blacklist_funnel_flg) AS funnel_sessions,
    SUM(
        CASE
            WHEN blacklist_funnel_flg = 1
             AND (
                 first_bonus_blacklist_dttm IS NULL
                 OR first_promo_blacklist_after_bonus_dttm IS NULL
                 OR first_promo_blacklist_after_bonus_dttm <= first_bonus_blacklist_dttm
             )
            THEN 1
            ELSE 0
        END
    ) AS incorrect_flags
FROM logs_features;
"""

pd.read_sql_query(query, sql_conn)


,total_sessions,funnel_sessions,incorrect_flags
0,37593,1200,0


**Результат проверки.** Таблица `logs_features` содержит **37 593 сессии**. Для **1 200 сессий** `blacklist_funnel_flg = 1`. В выполненной проверке не нашлось флагов 1 с отсутствующими датами или нарушенным порядком событий.

Это проверка нескольких условий, а не доказательство правильности абсолютно всех строк.


## 5. Признаки по заказам: `02_orders.sql`

Теперь проверю второй SQL-файл. Он рассчитывает количество оплаченных заказов (`paid` или `completed`), количество оплаченных заказов **без промокода**, их сумму (GMV) и дату первого такого заказа.

В рабочей копии исправила синтаксис и добавила расчёт GMV. Запущу файл и посмотрю результат.


In [11]:

# Запускаем исправленный SQL-файл
sql_file = project / "sql" / "02_orders.sql"

sql_conn.executescript(
    sql_file.read_text(encoding="utf-8")
)

# Проверяем результат
query = """
SELECT *
FROM orders_features
ORDER BY customer_id, session_id
LIMIT 10;
"""

pd.read_sql_query(query, sql_conn)


,customer_id,session_id,paid_orders_cnt,paid_orders_without_promo_cnt,paid_orders_without_promo_gmv,first_paid_order_without_promo_dttm
0,100001,target-100001-1,1,1,15373.50,2026-07-17 06:04:00
1,100001,target-100001-2,1,1,23410.46,2026-07-02 00:54:59
2,100002,order-only-002823,0,0,0.00,None
3,100002,target-100002-1,1,1,22334.94,2026-07-19 05:24:00
4,100002,target-100002-2,1,1,6161.33,2026-07-01 18:16:59
5,100003,order-only-000874,1,0,0.00,None
6,100003,target-100003-1,1,1,14426.88,2026-07-17 19:16:00
7,100003,target-100003-2,1,1,11961.47,2026-07-01 00:24:59
8,100004,target-100004-1,1,1,16145.77,2026-07-13 13:34:00
9,100004,target-100004-2,1,1,11862.57,2026-07-27 07:28:59


**Наблюдение.** В некоторых сессиях есть оплаченные заказы, но нет подходящих заказов **без промокода**. Для них сумма `paid_orders_without_promo_gmv` равна нулю.

### Проверка расчётов по заказам

После запуска получилась таблица `orders_features`. Проверю три базовых условия: заказов без промокода не больше, чем всех оплаченных; GMV не отрицателен; для подходящих заказов есть дата первого заказа.


In [12]:

query = """
SELECT
    COUNT(*) AS total_sessions,

    SUM(
        CASE
            WHEN paid_orders_without_promo_cnt > paid_orders_cnt
            THEN 1 ELSE 0
        END
    ) AS incorrect_counts,

    SUM(
        CASE
            WHEN paid_orders_without_promo_gmv < 0
            THEN 1 ELSE 0
        END
    ) AS negative_gmv,

    SUM(
        CASE
            WHEN paid_orders_without_promo_cnt > 0
             AND first_paid_order_without_promo_dttm IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_dates

FROM orders_features;
"""

orders_check = pd.read_sql_query(query, sql_conn)

orders_check


,total_sessions,incorrect_counts,negative_gmv,missing_dates
0,8000,0,0,0


**Результат проверки.** Получилось **8 000 сессий с записями о заказах**. Во всех трёх проверках — **0 нарушений**.

Эти проверки полезны, но не заменяют сверку SQL-логики с исходными заказами.


## 6. Итоговые признаки: `03_customer_features.sql`

Последний SQL-файл объединяет логи и заказы, а затем суммирует показатели **по пользователям**. Для этого используются `UNION`, `LEFT JOIN` и `COALESCE` — чтобы сохранить клиентов, даже если у них нет событий или заказов.

В рабочей копии исправила синтаксис и заполнила две формулы: количество сессий с конверсией и долю таких сессий среди прошедших blacklist-воронку. Теперь проверю размер итоговой таблицы.


In [13]:

# Открываем итоговый SQL-файл
sql_file = project / "sql" / "03_customer_features.sql"

# Выполняем SQL-запрос
sql_conn.executescript(
    sql_file.read_text(encoding="utf-8")
)

# Загружаем итоговую таблицу
customer_features = pd.read_sql_query(
    """
    SELECT *
    FROM customer_features
    ORDER BY customer_id;
    """,
    sql_conn
)

# Проверяем размер и первые строки
print("Размер итоговой таблицы:", customer_features.shape)

display(customer_features.head())


Размер итоговой таблицы: (5000, 9)


,customer_id,bonus_blacklist_requests_cnt,promo_blacklist_requests_cnt,blacklist_funnel_sessions_cnt,paid_orders_cnt,paid_orders_without_promo_cnt,paid_orders_without_promo_gmv,converted_blacklist_sessions_cnt,blacklist_to_order_conversion_rate
0,100001,6,2,2,2,2,38783.96,1,0.5
1,100002,3,2,2,2,2,28496.27,1,0.5
2,100003,4,2,2,3,2,26388.35,1,0.5
3,100004,4,3,2,2,2,28008.34,1,0.5
4,100005,4,2,2,2,2,28773.31,1,0.5


**Результат.** SQL создал таблицу нужной структуры: **5 000 пользователей и 8 признаков**. Осталось проверить ограничения на значения и отсутствие дубликатов.

### Проверка итоговой таблицы

Таблица `customer_features` содержит **5 000 строк и 9 столбцов**: идентификатор и восемь признаков.

Теперь проверю число уникальных пользователей, дубликаты, пропуски и допустимый диапазон конверсии от 0 до 1.


In [14]:

# Проверяем количество пользователей
print("Строк:", len(customer_features))

# Проверяем уникальность пользователей
print(
    "Уникальных пользователей:",
    customer_features["customer_id"].nunique()
)

# Проверяем дубликаты
print(
    "Дубликатов:",
    customer_features["customer_id"].duplicated().sum()
)

# Проверяем пропуски
print(
    "Пропусков:",
    customer_features.isna().sum().sum()
)

# Проверяем диапазон конверсии
invalid_conversion = ~customer_features[
    "blacklist_to_order_conversion_rate"
].between(0, 1)

print("Некорректных конверсий:", invalid_conversion.sum())


Строк: 5000
Уникальных пользователей: 5000
Дубликатов: 0
Пропусков: 0
Некорректных конверсий: 0


## 7. Итоги и дальнейшая проверка

Удалось восстановить расчёт трёх SQL-этапов: по логам, по заказам и на уровне пользователя. Итоговая таблица содержит **5 000 уникальных клиентов** и **8 признаков**. В выполненных проверках не обнаружены дубликаты, пропуски или значения конверсии вне диапазона от 0 до 1.

Основные приёмы, которые использовала: `WHERE`, `CASE WHEN`, `GROUP BY`, `json_each()`, `LEFT JOIN`, `COALESCE` и расчёт конверсии.
